# Notebook 03 — Transformación según reglas de negocio

Tercer sub-bloque del Tema 04. Con los DataFrames ya **limpios y tipados** del Notebook 02, ahora los **reorganizas** hacia el modelo dimensional: dimensiones desnormalizadas, `dim_date` generada en pandas, tabla de hechos con surrogate keys resueltas vía merge sucesivo.

Este notebook es el equivalente programático de los `INSERT…SELECT…JOIN` que pueblan `northwind_dwh` en los scripts SQL del repo. Al terminar deberías tener en memoria seis DataFrames listos para cargar al DWH (las 5 dims + la fact).

## Setup — recuperar DataFrames limpios del Notebook 02

> *Por publicar.*

Re-ejecución del engine + extracción + limpieza mínima. Notebook auto-contenido.

In [ ]:
# TODO: imports + engine + extracción + limpieza inicial

## Cálculo de valores derivados

> *Por publicar.*

Operaciones **vectorizadas** sobre columnas — el equivalente a las columnas calculadas (`extended_price`, `line_total`) del DDL:

- `df['total'] = df['quantity'] * df['unit_price']` — sin loops, pandas opera en bloque.
- Ratios, márgenes, descuentos aplicados.
- Conexión con lo del Tema 02: las columnas `GENERATED ALWAYS AS … STORED` del DDL son la versión declarativa de lo mismo.

In [ ]:
# TODO: cálculo de line_total = quantity * unit_price * (1 - discount)

## Joins entre DataFrames con `merge`

> *Por publicar.*

- Sintaxis: `pd.merge(left, right, on='col', how='inner'|'left'|'right'|'outer')`.
- Equivalencia con `JOIN` de SQL — mismas semánticas, distinta sintaxis.
- Múltiples keys: `on=['col1', 'col2']` o `left_on`/`right_on` cuando los nombres difieren.
- **Cuidado con duplicación de filas** si la key no es única en uno de los lados (mismo riesgo que `fan trap` en SQL).

In [ ]:
# TODO: ejemplo de merge: order_details + products + categories

## Generación de `dim_date` en pandas

> *Por publicar.*

La dimensión de fecha es **sintética** — no viene del OLTP, se genera. Patrón:

- `pd.date_range(start='1996-01-01', end='1998-12-31', freq='D')` para el rango necesario.
- Atributos derivados: `.dt.year`, `.dt.quarter`, `.dt.month`, `.dt.day_of_week`, `.dt.day_name(locale='es_ES.UTF-8')`.
- Smart key: `date_key = año * 10000 + mes * 100 + día` (formato `YYYYMMDD` como int).
- Bandera `is_weekend`.

Recordar del Tema 02: `dim_date` se **pre-popula densamente** — una fila por día del rango, exista o no actividad ese día.

In [ ]:
# TODO: generación de dim_date

## Construcción de las dimensiones desnormalizadas

> *Por publicar.*

Para cada dimensión del DWH:

- **`dim_customer`** — copia directa de `customers` con renombres y selección de columnas.
- **`dim_product`** — merge de `products + categories + suppliers` aplanado (esto es la desnormalización que vimos en el Tema 02).
- **`dim_employee`** — `employees` con `reports_to` resuelto a `reports_to_name` vía self-merge.
- **`dim_shipper`** — copia de `shippers`.

Surrogate key: asignar `*_key` como índice secuencial nuevo (1, 2, 3, …) **independiente** del ID natural.

In [ ]:
# TODO: dim_customer

In [ ]:
# TODO: dim_product (con merge a categories y suppliers)

In [ ]:
# TODO: dim_employee (con self-merge para reports_to_name)

In [ ]:
# TODO: dim_shipper

## Construcción de `fact_sales`

> *Por publicar.*

La fact es el último paso porque depende de que **todas las dimensiones ya tengan sus surrogate keys asignadas**. Patrón:

1. Partir de `order_details` joineado con `orders` (para fechas y FKs).
2. **Merge sucesivo** con cada dimensión para resolver `customer_id → customer_key`, `product_id → product_key`, etc.
3. Manejar las **tres FKs de fecha** (role-playing): `order_date_key`, `required_date_key`, `shipped_date_key` — tres merges con `dim_date`.
4. Mantener el `order_id` como **degenerate dimension**.
5. Generar el `sale_key` secuencial al final.

In [ ]:
# TODO: construcción de fact_sales con merge sucesivo

## Validación de integridad referencial

> *Por publicar.*

**Antes de cargar al DWH**, verificar:

- Ninguna FK de la fact tiene `NaN` (sería un cliente/producto/fecha sin matchear).
- Conteos de filas tienen sentido (`len(fact_sales)` ≈ `len(order_details)`).
- No hay surrogate keys duplicadas en las dimensiones.
- Las medidas calculadas (`line_total`) coinciden con valores spot-checked manualmente.

Si algo falla aquí, **es más barato corregirlo en memoria** que después de cargar al DWH.

In [ ]:
# TODO: validaciones de integridad referencial

## Cierre

> *Por publicar.*

Tienes seis DataFrames en memoria — las 5 dims + `fact_sales` — con surrogate keys resueltas y validados. El siguiente notebook (**04 — Carga y orquestación**) los lleva al DWH y empaqueta todo el pipeline en un script productivo.

---

<p align="center">
<a href="02_limpieza_y_perfilado.ipynb">← Anterior: Notebook 02</a> | <a href="Readme.md">Volver al índice</a> | <a href="04_carga_y_orquestacion.ipynb">Siguiente: Notebook 04 — Carga y orquestación →</a>
</p>